Для начала стоит разобраться с проблемой комутативности токенов. Попытаемся понять сколько вообще бракованных выражений на данный момент есть в датасете.

In [2]:
from pathlib import Path
import numpy as np
import sympy as sp
import gzip
import pickle
import os
import sys
from tqdm.auto import tqdm

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.tokenizer import Tokenizer

BASE_DIR = Path("../").resolve()
DATA_DIR = BASE_DIR / "data" / "train"


In [7]:
tokenizer = Tokenizer()


def load_chunk(filepath):
    with gzip.open(filepath, "rb") as f:
        data = pickle.load(f)
        
        errors_counts = 0

        for item in data:
            tokens = item["tokens"]
            expr = tokenizer.token_seq_to_expr(tokens)
            new_expr = tokenizer.canonicalize_tree_structure(expr)
            new_tokens = tokenizer.expr_to_token_seq(new_expr)

            if not np.array_equal(tokens, new_tokens):
                errors_counts += 1

                if (errors_counts < 10):
                    print(f"Проблема в {item['expr_str']}")
                    print("Изначальное выражение выглядело как:")
                    sp.print_tree(expr, assumptions=False)
                    print("Его последовательность токенов:")
                    print(tokens)
                    print("Вот так выглядит канонизированная версия:")
                    sp.print_tree(new_expr, assumptions=False)
                    print("Его последовательность токенов:")
                    print(new_tokens)

                

    del data
    print(f"Всего проблем: {errors_counts}") 

In [8]:
# load_chunk("/home/yauheni/pet_project/Graph2Eq/data/train/chunk_0.pkl.gz")

Для решения просто пересоберем каждое дерево в канонический вид. Для этого отсортируем аргументы каждой Mul и Add, а также перестроим дерево, так чтобы оно стало правым бинарным. Таким образом для одного и того же выраения последовательность токенов будет ДЕТЕРМЕНИРОВАННОЙ!

In [9]:
from sympy.core.cache import clear_cache
from src.tokenizer import InvalidExpressionError

def replace_cos_unevaluated(expr):
    """
    Рекурсивно обходит дерево выражения и заменяет все sp.cos на sp.sin,
    не вызывая автоматического упрощения (evaluate=False).
    """
    if not expr.args:
        return expr
    new_args = [replace_cos_unevaluated(arg) for arg in expr.args]
    current_func = expr.func
    if current_func == sp.cos:
        new_func = sp.sin
    else:
        new_func = current_func
    return new_func(*new_args, evaluate=False)


def clean_item(item):
    try:
        
        tokens = item["tokens"]
        expr = tokenizer.token_seq_to_expr(tokens)


        has_sin = expr.has(sp.sin)
        has_cos = expr.has(sp.cos)

        if has_sin and has_cos:
            return None

        if has_cos:
            processed_expr = replace_cos_unevaluated(expr)
        else:
            # Если cos нет, оставляем выражение как есть.
            processed_expr = expr

        clear_expr = tokenizer.canonicalize_tree_structure(processed_expr)
        new_tokens = tokenizer.expr_to_token_seq(clear_expr)
        


        return {
                'expr_str': str(clear_expr), 
                'expr_instantiated_str': item['expr_instantiated_str'], 
                'tokens': new_tokens,
                'points': item['points']
            }
    
    except InvalidExpressionError as e:
        print(f"Пропуск элемента из-за ошибки токенизации: {item['expr_str']}. Причина: {e}")
        return None

In [10]:
import glob
import os
from sympy.core.cache import clear_cache

input_dir = DATA_DIR
output_dir = DATA_DIR.parent / "train_cleaned"
os.makedirs(output_dir, exist_ok=True)

CHUNK_SIZE = 50000

# Получаем отсортированный список старых чанков
input_files = sorted(glob.glob(os.path.join(input_dir, "chunk_*.pkl.gz")))

if not input_files:
    print("В папке data/train не найдено чанков для обработки.")
else:
    cleaned_buffer = []
    new_chunk_index = 0
    total_processed_count = 0
    total_skipped_count = 0
    
    # Счетчик для очистки кэша SymPy
    operations_count = 0 

    print(f"Найдено {len(input_files)} чанков. Начинаю пересборку...")

    # Создаем прогресс-бар, который будет считать выражения
    with tqdm(desc="Обработка выражений", unit=" expr") as pbar:
        for filepath in input_files:
            with gzip.open(filepath, "rb") as f:
                data = pickle.load(f)
            
            # Показываем, какой файл сейчас читаем
            pbar.set_postfix(file=os.path.basename(filepath), skipped=total_skipped_count)
            
            for item in data:
                # --- 1. СПАСАЕМ ОПЕРАТИВНУЮ ПАМЯТЬ ---
                operations_count += 1
                if operations_count % 10000 == 0:
                    clear_cache()  

                cleaned = clean_item(item)
                if cleaned:
                    cleaned_buffer.append(cleaned)
                else:
                    total_skipped_count += 1
                
                if len(cleaned_buffer) >= CHUNK_SIZE:
                    chunk_to_save = cleaned_buffer[:CHUNK_SIZE]
                    
                    # --- 2. БЕЗОПАСНОЕ СОХРАНЕНИЕ ---
                    savepath = os.path.join(output_dir, f"chunk_{new_chunk_index}.pkl.gz")
                    tmp_savepath = os.path.join(output_dir, f"chunk_{new_chunk_index}.tmp.gz")
                    
                    with gzip.open(tmp_savepath, "wb") as f_out:
                        pickle.dump(chunk_to_save, f_out)
                    os.rename(tmp_savepath, savepath)
                    
                    tqdm.write(f"Сохранен новый чанк {new_chunk_index} ({len(chunk_to_save)} элементов)")
                    
                    total_processed_count += len(chunk_to_save)
                    cleaned_buffer = cleaned_buffer[CHUNK_SIZE:]
                    new_chunk_index += 1
                
                # Обновляем прогресс-бар на 1 выражение
                pbar.update(1)
                
            pbar.set_postfix(file=os.path.basename(filepath), skipped=total_skipped_count)

    if cleaned_buffer:
        savepath = os.path.join(output_dir, f"chunk_{new_chunk_index}.pkl.gz")
        tmp_savepath = os.path.join(output_dir, f"chunk_{new_chunk_index}.tmp.gz")
        
        with gzip.open(tmp_savepath, "wb") as f_out:
            pickle.dump(cleaned_buffer, f_out)
        os.rename(tmp_savepath, savepath)
        
        total_processed_count += len(cleaned_buffer)
        tqdm.write(f"Сохранен последний чанк {new_chunk_index} ({len(cleaned_buffer)} элементов)")

    print("\n=== ИТОГ ===")
    print(f"Успешно очищено и сохранено: {total_processed_count} выражений.")
    print(f"Всего создано новых чанков: {new_chunk_index + (1 if cleaned_buffer else 0)}")
    print(f"Пропущено из-за ошибок: {total_skipped_count} выражений.")

Найдено 59 чанков. Начинаю пересборку...


Обработка выражений: 0 expr [00:00, ? expr/s]

Сохранен новый чанк 0 (50000 элементов)
Сохранен новый чанк 1 (50000 элементов)
Сохранен новый чанк 2 (50000 элементов)
Сохранен новый чанк 3 (50000 элементов)
Сохранен новый чанк 4 (50000 элементов)
Сохранен новый чанк 5 (50000 элементов)
Сохранен новый чанк 6 (50000 элементов)
Сохранен новый чанк 7 (50000 элементов)
Сохранен новый чанк 8 (50000 элементов)
Сохранен новый чанк 9 (50000 элементов)
Сохранен новый чанк 10 (50000 элементов)
Сохранен новый чанк 11 (50000 элементов)
Сохранен новый чанк 12 (50000 элементов)
Сохранен новый чанк 13 (50000 элементов)
Сохранен новый чанк 14 (50000 элементов)
Сохранен новый чанк 15 (50000 элементов)
Сохранен новый чанк 16 (50000 элементов)
Сохранен новый чанк 17 (50000 элементов)
Сохранен новый чанк 18 (50000 элементов)
Сохранен новый чанк 19 (50000 элементов)
Сохранен новый чанк 20 (50000 элементов)
Сохранен новый чанк 21 (50000 элементов)
Сохранен новый чанк 22 (50000 элементов)
Сохранен новый чанк 23 (50000 элементов)
Сохранен новый чанк 24 (50

Теперь нету проблем с комутативностью, ассоициативностью и дистрибутивностью :)

Теперь поймём, что на самом деле существует следующая проблема. Модель будет пытаться угадывать форму графика в самом наиобщем виде ставя плейсхолдеры для констант. Заметим, что у нас есть токены под $sin$ и $cos$. Но ведь формы $sin(C \cdot Arg + C)$ и $cos(C \cdot Arg + C)$ отличаются только значением констант на которые модели должно быть всё равно. Таким образом имеет смысл запретить модели вообще предсказывать токен $cos$ и оставить в таргетах только токен $sin$

In [11]:
def check_trigonometry_chunk(filepath):
    with gzip.open(filepath, "rb") as f:
        data = pickle.load(f)
        
        trig_counts = 0

        for item in data:
            tokens = item["tokens"]
            if 6 in tokens or 11 in tokens:
                trig_counts += 1
                if (trig_counts < 10):
                    print(item['expr_str'])

In [13]:
check_trigonometry_chunk("/home/yauheni/pet_project/Graph2Eq/data/train_cleaned/chunk_0.pkl.gz")

C*(log(C*x + C)*sin(C*x + C))
C*sin(C*x + C)
C*sin(C*(log(C*x + C)/(C*x + C)) + C)
C + (C*x + C*log(C + (C*x + C*sin(C*x + C))))
C*sin(C*x + C)
C*sin(C*x + C) + C
C*sin(C*sin(C/(C*x + C)))
C + (C*x + (C*x**2 + (C*(x*sin(C*x + C)) + (C*sin(C*x + C)**2 + C*sin(C*x + C)))))
C + (C*log(C + (C*x**2 + C*x)) + C*sin(C*x + C))


Как видим везде внутри sin и cos действительно стоит C. Таким образом логичным решением кажется просто избавиться от токена cos.